# 第1章 深度学习导论 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter1/Chapter1.md]]
- [[课程/深度学习/课堂笔记/2026-08-31-第1节-转写_电子科大王君豪的快速会议.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## T1-B 双特征响应模型与 NumPy／PyTorch 交叉核验

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]线性回归与Tensor属性；[[课程/深度学习/课堂笔记/2026-08-31-第1节-转写_电子科大王君豪的快速会议.md]]第四、六节；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C1-09。新编模拟题。


**总分100，30分钟。** 一道完整题，从空白开始。区别于测试A，本题是双特征拟合并作张量交叉核验。

1. **数据（20分）**：`rng=np.random.default_rng(20261101)`；依次生成训练 `X=rng.uniform(-1,1,(30,2))` 和测试 `Xt=rng.uniform(-1,1,(16,2))`。目标 `f(Z)=0.5*Z[:,0]-1.2*Z[:,1]+2`；训练标签 `y=f(X)+0.08*np.cos(4*X[:,0]+X[:,1])`，测试标签 `yt=f(Xt)`（干净目标，不含扰动）。保存 `response_X,response_y,response_Xt,response_yt`。
2. **拟合接口（30分）**：实现 `fit_response(X,y,Xt)`，以 `[w1,w2,b]` 顺序返回 `(beta,train_pred,test_pred)`。使用最小二乘，不求逆。调用后保存 `response_coef,response_train_pred,response_test_pred`。
3. **跨库核验（20分）**：把测试特征与权重转成CPU float64张量，用矩阵乘法和偏置作预测，保存 `response_torch_pred`；打印与NumPy预测的最大绝对差。不得将NumPy预测直接转换后冒充Torch前向计算。
4. **评价（15分）**：保存并打印 `response_train_rmse,response_test_rmse,response_gap_abs,response_gap_signed`，差距分别为课件绝对泛化差距与额外有符号差。注明前者训练误差对含扰动标签、测试误差对干净标签计算，不能从有符号差的正负直接断言过拟合。
5. **图和解释（15分）**：创建本地输出目录，画测试 `yt` 与预测的对应点及 `y=x` 参考线，保存 `output/Q101B.png`；解释偏置列的作用与两种标签口径的区别。

数值容差 `atol=1e-10`；除已在教学出现的设备属性、绘图管理与路径辅助外，不需表外模型API。


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
def fit_response(X,y,Xt):
    A=np.column_stack((X,np.ones_like(X[:,0])))
    At=np.column_stack((Xt,np.ones_like(Xt[:,0])))
    b=np.linalg.lstsq(A,y,rcond=None)[0]
    return b,A@b,At@b
_rng=np.random.default_rng(20261101)
response_X=_rng.uniform(-1,1,(30,2)); response_Xt=_rng.uniform(-1,1,(16,2))
_f=lambda Z:.5*Z[:,0]-1.2*Z[:,1]+2
response_y=_f(response_X)+.08*np.cos(4*response_X[:,0]+response_X[:,1])
response_yt=_f(response_Xt)
response_coef,response_train_pred,response_test_pred=fit_response(response_X,response_y,response_Xt)
response_torch_pred=torch.tensor(response_Xt,dtype=torch.float64,device='cpu') @ torch.tensor(response_coef[:2],dtype=torch.float64,device='cpu')+float(response_coef[2])
print('NumPy/Torch max difference:',np.max(np.abs(response_test_pred-response_torch_pred.numpy())))
response_train_rmse=float(np.sqrt(np.mean((response_train_pred-response_y)**2)))
response_test_rmse=float(np.sqrt(np.mean((response_test_pred-response_yt)**2)))
response_gap_signed=response_test_rmse-response_train_rmse
response_gap_abs=abs(response_gap_signed)
print('coef:',response_coef)
print('train_RMSE_noisy test_RMSE_clean gap_abs gap_signed')
print(response_train_rmse,response_test_rmse,response_gap_abs,response_gap_signed)
Path('output').mkdir(exist_ok=True)
plt.figure()
plt.plot(response_yt,response_test_pred,'o',label='test prediction')
_a=np.array([.2,3.8]); plt.plot(_a,_a,'k--',label='ideal')
plt.xlabel('clean target'); plt.ylabel('prediction'); plt.legend()
plt.savefig('output/Q101B.png'); plt.close()
print('The constant column learns a nonzero intercept.')
print('Training includes a deterministic perturbation, whereas testing targets the clean response.')
print('This difference in label noise affects the signed gap; its sign alone does not prove overfitting.')


### 解析与点评

本题有意采用“扰动训练标签／干净测试标签”，因此必须给指标加清楚语义。交叉库计算一致只能说明前向实现一致，不能证明模型对所有数据正确。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
_rng=np.random.default_rng(20261101)
_X=_rng.uniform(-1,1,(30,2)); _Xt=_rng.uniform(-1,1,(16,2))
assert np.allclose(response_X,_X) and np.allclose(response_Xt,_Xt)
assert np.allclose(response_y,.5*_X[:,0]-1.2*_X[:,1]+2+.08*np.cos(4*_X[:,0]+_X[:,1]))
assert np.allclose(response_yt,.5*_Xt[:,0]-1.2*_Xt[:,1]+2)
_b,_p,_t=fit_response(np.array([[0.,0.],[1.,0.],[0.,1.],[1.,1.]]),np.array([2.,2.5,.8,1.3]),np.array([[2.,-1.]]))
assert np.allclose(_b,[.5,-1.2,2],atol=1e-10) and np.allclose(_t,[4.2],atol=1e-10)
_A=np.column_stack((response_X,np.ones(30)))
assert np.linalg.norm(_A.T@(response_train_pred-response_y)) < 1e-9
assert response_torch_pred.dtype == torch.float64 and response_torch_pred.device.type == 'cpu'
assert np.allclose(response_torch_pred.numpy(),response_test_pred,atol=1e-10)
assert np.isclose(response_train_rmse,np.sqrt(np.mean((response_train_pred-response_y)**2)))
assert np.isclose(response_test_rmse,np.sqrt(np.mean((response_test_pred-response_yt)**2)))
assert np.isclose(response_gap_abs,abs(response_test_rmse-response_train_rmse))
assert np.isclose(response_gap_signed,response_test_rmse-response_train_rmse)
assert Path('output/Q101B.png').stat().st_size > 1000

print("自检通过")
